<font size="+1" color='#FFD700'><b>Published on September 8, 2024</b></font>
  
<font size="+1" color='#4B0082'><i>Author: Jocelyn C. Dumlao</i></font>

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007FFF;font-family:cursive;overflow:hidden"><p style="padding:15px;color:white;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b>21 Chatbots: Evaluating Their Impact on University Learning</p></div>

![image](https://storage.googleapis.com/kaggle-datasets-images/5631046/9300150/258897a3b706bf2374c91b0d5b7efdb1/dataset-cover.png?t=2024-09-02-10-16-24)

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007FFF;font-family:cursive;overflow:hidden"><p style="padding:15px;color:white;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b> Introduction</p></div>

This study explores how conversational chatbots impact university students' electronic learning in Bulgaria. Between May 14 and May 31, 2023, an online survey gathered 131 responses from students using Google Forms questionnaire distributed via email and social media. With advancements in AI and natural language processing, chatbots have become popular in education. Most respondents (89%) had used chatbots as learning tools. The survey assessed their experience with chatbots, focusing on use frequency, usefulness, trust, and overall conditions. The second part of the study examines how well these chatbots, particularly in solving mathematics problems,support university-level learning. Among the seven chatbots tested, ChatGPt Plus showed the best performance.

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007FFF;font-family:cursive;overflow:hidden"><p style="padding:15px;color:white;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b>Import Modules</p></div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007FFF;font-family:cursive;overflow:hidden"><p style="padding:15px;color:white;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b>Load the Dataset</p></div>


In [ ]:
# Load dataset with delimiter (semicolon)
df = pd.read_csv(
    '/kaggle/input/chatbots-impact-on-university-learning/Impact of Conversational Chatbots on Learning of University Students/AI_Chatbots_Students_Attitude_Dataset_EN.csv',
    encoding='ISO-8859-1',
    encoding_errors='replace',
    delimiter=';'  # Set delimiter to semicolon
)

# Check the first few rows and column names to confirm structure
df.columns  # Check column names to know what to use


In [ ]:
df.head()

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007FFF;font-family:cursive;overflow:hidden"><p style="padding:15px;color:white;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b>Exploratory Data Analysis (EDA)📊</p></div>


In [ ]:
# Convert Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Group data by Q1 (Education Level), Q3 (Gender), and Q4 (Frequency of response)
grouped_df = df.groupby(['Q1', 'Q4']).agg('count')

# Plot heatmap of the response counts across education levels and frequency
plt.figure(figsize=(14, 6))
pivot_table = df.pivot_table(index='Q1', columns='Q4', values='Q5.1', aggfunc='count')
sns.heatmap(pivot_table, annot=True, cmap='coolwarm', fmt="g")
plt.title('Response Count Heatmap for Education Level and Frequency of Responses')
plt.xlabel('Frequency (Q4)')
plt.ylabel('Education Level (Q1)')
plt.show()



In [ ]:
# Visualize agreement levels (Q5.1 - Q5.5) using a stacked bar chart
# Aggregate the data for visualization
response_counts = df[['Q1', 'Q4', 'Q5.1', 'Q5.2', 'Q5.3', 'Q5.4', 'Q5.5']].melt(
    id_vars=['Q1', 'Q4'], value_vars=['Q5.1', 'Q5.2', 'Q5.3', 'Q5.4', 'Q5.5'], 
    var_name='Question', value_name='Response'
)

# Check if the melt operation was successful
print(response_counts.head())

# Plot stacked bar chart of response types
plt.figure(figsize=(14, 6))
response_order = ['Strongly Disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly Agree']
sns.countplot(data=response_counts, x='Question', hue='Response', 
              order=['Q5.1', 'Q5.2', 'Q5.3', 'Q5.4', 'Q5.5'], 
              hue_order=response_order, palette='coolwarm')

plt.title('Distribution of Agreement Levels Across Questions (Q5.1 to Q5.5)')
plt.xlabel('Questions (Q5.1 to Q5.5)')
plt.ylabel('Count of Responses')
plt.legend(title='Response Level')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Analyze and visualize response trends for different demographic groups
# For example, looking at how often respondents from different educational levels select "Agree"
agree_responses = response_counts[response_counts['Response'] == 'Agree'].groupby(['Q1', 'Q4']).size().unstack()

agree_responses.plot(kind='bar', stacked=True, colormap='viridis', figsize=(14, 6))
plt.title('Agreement Responses by Education Level and Frequency')
plt.xlabel('Education Level and Frequency (Q4)')
plt.ylabel('Number of "Agree" Responses')
plt.xticks(rotation=0)
plt.legend(title='Frequency (Q4)', loc='upper right')
plt.show()

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007FFF;font-family:cursive;overflow:hidden"><p style="padding:15px;color:white;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b>Natural Language Processing Analysis (NLP)</p></div>

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#4B0082;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>1. </b>Sentiment Analysis of Responses</p></div>
 
- Analyze the sentiment of responses for each question and visualize the sentiment distribution.

In [ ]:
import pandas as pd
from textblob import TextBlob
import matplotlib.pyplot as plt


# Combine responses for sentiment analysis
responses = df[['Q1', 'Q2', 'Q3', 'Q4', 'Q5.1', 'Q5.2', 'Q5.3', 'Q5.4', 'Q5.5', 'Q8.2', 'Q8.3', 'Q8.4', 'Q8.5', 'Q9.1', 'Q9.2', 'Q9.3', 'Q9.4', 'Q9.5', 'Q10']].fillna('')

# Function to calculate sentiment
def get_sentiment(text):
    analysis = TextBlob(text)
    return analysis.sentiment.polarity

# Calculate sentiment for each response
sentiments = responses.apply(lambda x: x.apply(get_sentiment).mean(), axis=1)

# Plot sentiment distribution
plt.figure(figsize=(14, 6))
plt.hist(sentiments, bins=20, edgecolor='k', alpha=0.7)
plt.title('Sentiment Distribution of Responses')
plt.xlabel('Sentiment Score')
plt.ylabel('Frequency')
#plt.grid(True)
plt.show()


- Combines all responses into a single text per entry, calculated the sentiment score using TextBlob, and visualizes the distribution of sentiment scores. The histogram shows how positive or negative the responses are on average.

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#4B0082;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>2. </b>Word Cloud of Most Common Terms</p></div>
 
- Generate a word cloud to visualize the most frequent terms in the responses.

In [ ]:
from wordcloud import WordCloud

# Combine all responses into one large text
text = ' '.join(df[['Q1', 'Q2', 'Q3', 'Q4', 'Q5.1', 'Q5.2', 'Q5.3', 'Q5.4', 'Q5.5', 'Q8.2', 'Q8.3', 'Q8.4', 'Q8.5', 'Q9.1', 'Q9.2', 'Q9.3', 'Q9.4', 'Q9.5', 'Q10']].fillna('').astype(str).agg(' '.join, axis=1))

# Generate the word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

# Plot the word cloud
plt.figure(figsize=(18, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Responses')
plt.show()


- A word cloud to visually represent the most common terms in the responses. Frequently occurring words will appear larger in the word cloud, providing insights into common themes or topics.

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#4B0082;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>3. </b>Topic Modeling with Latent Dirichlet Allocation (LDA)</p></div> 

- Identify topics in the responses using LDA and visualize the most important words for each topic.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import seaborn as sns

# Combine all responses into one large text
text = df[['Q1', 'Q2', 'Q3', 'Q4', 'Q5.1', 'Q5.2', 'Q5.3', 'Q5.4', 'Q5.5', 'Q8.2', 'Q8.3', 'Q8.4', 'Q8.5', 'Q9.1', 'Q9.2', 'Q9.3', 'Q9.4', 'Q9.5', 'Q10']].fillna('').astype(str).agg(' '.join, axis=1)

# Vectorize the text data
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(text)

# Apply LDA
lda = LatentDirichletAllocation(n_components=5, random_state=0)
lda.fit(X)

# Display topics
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic #{topic_idx}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))
    print()
    
no_top_words = 10
display_topics(lda, vectorizer.get_feature_names_out(), no_top_words)

# Visualize the topics
topic_words = []
for topic_idx, topic in enumerate(lda.components_):
    words = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-no_top_words - 1:-1]]
    topic_words.append(words)

# Create a DataFrame for visualization
topics_df = pd.DataFrame(topic_words, columns=[f'Word {i+1}' for i in range(no_top_words)])
topics_df.index.name = 'Topic'
topics_df = topics_df.reset_index()



In [ ]:
# Define the number of words to display
no_top_words = 10

# Extract words and their corresponding topic weight (importance)
topic_word_weights = []
for topic_idx, topic in enumerate(lda.components_):
    words = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-no_top_words - 1:-1]]
    weights = [topic[i] for i in topic.argsort()[:-no_top_words - 1:-1]]
    topic_word_weights.append(pd.DataFrame({'Words': words, 'Weights': weights}))

# Plot top words for each topic
for i, topic_df in enumerate(topic_word_weights):
    plt.figure(figsize=(12, 4))
    sns.barplot(x='Weights', y='Words', data=topic_df, palette='YlGnBu')
    plt.title(f'Topic {i+1} - Top Words')
    plt.xlabel('Weight')
    plt.ylabel('Word')
    plt.show()


- Words and Weights Extraction: We extract the top m words and their corresponding weights(importance values) from the LDA model's components for each topic.
- Bar Chart Visualization: For each topic, we create a bar chart showing the top words and their importance for the topic.
- Matplotlib and Seaborn: The sns.barplot() functions is used to plot the words and their corresponding importance weights in a visually appealing manner.

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#4B0082;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>4. </b>Response Length Analysis</p></div> 

- Analyze the length of responses to each question and visualize the distribution.

In [ ]:
# Calculate length of responses for each question
response_lengths = df[['Q1', 'Q2', 'Q3', 'Q4', 'Q5.1', 'Q5.2', 'Q5.3', 'Q5.4', 'Q5.5', 'Q8.2', 'Q8.3', 'Q8.4', 'Q8.5', 'Q9.1', 'Q9.2', 'Q9.3', 'Q9.4', 'Q9.5', 'Q10']].fillna('').applymap(len)

# Plot response length distribution for each question
plt.figure(figsize=(14, 8))
for col in response_lengths.columns:
    plt.hist(response_lengths[col], bins=20, alpha=0.5, label=col)

plt.title('Distribution of Response Lengths for Each Question')
plt.xlabel('Response Length')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()


- Calculates the length of responses for each question and visualizes the distribution. The historgram shows how the length of responses varies across different questions, which can help in undestanding the depth of responses.

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007FFF;font-family:cursive;overflow:hidden"><p style="padding:15px;color:white;overflow:hidden;font-size:100%;letter-spacing:0.5px;margin:0"><b> </b>Chatbot</p></div>

# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 1:  </b>Timestamp Respone Bot</p></div> 

- This bot checks the time when the user completed the survey based on `Timestamp` and responds with an appropriate greeting.

In [ ]:
def chatbot_timestamp(timestamp):
    # Assuming the timestamp is provided in the format MM.DD.YYYY HH:MM:SS
    hour = int(timestamp.split()[1].split(":")[0])  # Extracting hour from the timestamp

    if hour < 12:
        return "Good morning! Thanks for completing the survey early."
    elif 12 <= hour < 18:
        return "Good afternoon! Hope you're having a productive day."
    else:
        return "Good evening! Thank you for taking the time to complete the survey."
        
# Example test
timestamp = "09.08.2024 14:30:00"  # Replace this with any timestamp in the correct format
message = chatbot_timestamp(timestamp)
print(message)


In [ ]:
#def chatbot_timestamp():
#    timestamp = input("Please enter the time you completed the survey (in the format MM.DD.YYYY HH:MM:SS): ").strip()
#    hour = int(timestamp.split()[1].split(":")[0])  # Extracting hour from the timestamp

#    if hour < 12:
#        print("Good morning! Thanks for completing the survey early.")
#    elif 12 <= hour < 18:
#        print("Good afternoon! Hope you're having a productive day.")
#    else:
#        print("Good evening! Thank you for taking the time to complete the survey.")
        
# Testing the bot
#chatbot_timestamp()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 2:  </b>Degree-Based Custom Feedback Bot</p></div> 

- This bot gives feedback based on the user's degree from `Q1` (Bachelor/Master)

In [ ]:
def chatbot_degree_feedback():
    degree = input("What is your academic degree? (Bachelor/Master): ").strip()

    if degree.lower() == "bachelor":
        print("As a Bachelor’s student, you're at the start of your academic journey. AI tools can assist you in building strong foundations.")
    elif degree.lower() == "master":
        print("As a Master’s student, you're diving deeper into research. AI can help streamline your advanced studies.")
    else:
        print("It looks like you entered an invalid degree. Please enter 'Bachelor' or 'Master'.")
        
# Testing the bot
chatbot_degree_feedback()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 3:  </b>Field of Study Bot</p></div> 

- This bot recommends AI tools based on the user's field of study (from `Q2`).

In [ ]:
def chatbot_field_of_study():
    field = input("What is your field of study? ").strip().lower()

    if "economic" in field:
        print("AI tools such as predictive modeling can be very useful in economic research and analysis.")
    else:
        print(f"Your field of study, {field}, may benefit from AI in research assistance and automation of routine tasks.")
        
# Testing the bot
chatbot_field_of_study()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 4:  </b>AI Usage Frequency and Motivation Bot</p></div> 

- This bot checks how frequently the user uses AI tools (`Q4`) and provides motivational advice.

In [ ]:
def chatbot_ai_usage_motivation():
    frequency = input("How often do you use AI tools? (Never, Rarely, Sometimes, Often): ").strip().lower()

    if frequency == "never":
        print("Don't hesitate to explore AI tools! They can greatly assist your learning process.")
    elif frequency == "rarely":
        print("Using AI tools a little more can enhance your research and productivity.")
    elif frequency == "sometimes":
        print("You're on the right path! Keep using AI tools to further improve your workflow.")
    elif frequency == "often":
        print("You seem to be familiar with AI tools. Have you considered exploring more advanced AI applications?")
    else:
        print("Please enter a valid frequency (Never, Rarely, Sometimes, Often).")
        
# Testing the bot
chatbot_ai_usage_motivation()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 5:  </b> Survey Sentiment Summary Bot</p></div> 

- This chatbot summarizes the overall sentiment of the user's answers from the `Q5.x` columns

In [ ]:
def chatbot_survey_sentiment():
    sentiment = input("How would you describe your attitude towards AI in education? (Strongly Agree, Agree, Neutral, Disagree, Strongly Disagree): ").strip().lower()

    if sentiment == "strongly agree":
        print("You seem very positive about AI! It's great to see you embracing new technologies.")
    elif sentiment == "agree":
        print("You have a favorable opinion of AI. It’s a valuable tool in modern education.")
    elif sentiment == "neutral":
        print("You are undecided about AI. Perhaps exploring more AI tools will help you form an opinion.")
    elif sentiment == "disagree":
        print("It’s important to address your concerns. Understanding how AI complements education may change your view.")
    elif sentiment == "strongly disagree":
        print("You may have strong concerns about AI. Engaging with how AI can be used responsibly might help alleviate these worries.")
    else:
        print("Please enter a valid sentiment (Strongly Agree, Agree, Neutral, Disagree, Strongly Disagree).")
        
# Testing the bot
chatbot_survey_sentiment()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 6:  </b>  Personalized AI Tool Suggestions Bot</p></div> 

- This bot suggests AI tools based on how often the user agrees or disagrees with AI tool usage for `Q5.x` columns.

In [ ]:
def chatbot_tool_suggestions():
    response = input("How would you rate your experience with AI tools? (Agree, Neutral, Disagree): ").strip().lower()

    if response == "agree":
        print("We recommend tools like GPT-3 for content generation and machine learning platforms like Google Colab for experiments.")
    elif response == "neutral":
        print("You might want to try basic AI tools like ChatGPT or Grammarly to see their potential.")
    elif response == "disagree":
        print("You might not have had the best experience. Consider trying different AI tools that align with your needs.")
    else:
        print("Please provide a valid response (Agree, Neutral, Disagree).")
        
# Testing the bot
chatbot_tool_suggestions()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 7:  </b> AI Ethical Concerns Bot</p></div> 

- This chatbot asks the user about their concerns over AI ethics and data privacy, based on their agreement level in the survey (related to `Q6`).

In [ ]:
def chatbot_ethics_concerns():
    concern = input("Are you concerned about AI ethics or data privacy? (Yes/No): ").strip().lower()

    if concern == "yes":
        print("AI ethics is a critical issue. It’s important to stay informed and ensure ethical practices in AI development.")
    elif concern == "no":
        print("It’s great that you’re confident in AI. However, staying updated on AI ethics will keep you aware of its impact.")
    else:
        print("Please enter a valid response (Yes/No).")
        
# Testing the bot
chatbot_ethics_concerns()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 8:  </b> AI in Future Careers Bot</p></div> 

- This chatbot discusses how AI might play a role in the user's future career based on their confidence in using AI tools (`Q7`).

In [ ]:
def chatbot_ai_careers():
    confident = input("Do you see AI playing a role in your future career? (Yes/No): ").strip().lower()

    if confident == "yes":
        print("AI is becoming integral to many industries. Stay curious and keep building your AI skills!")
    elif confident == "no":
        print("Even if AI doesn’t seem relevant now, it’s likely to be part of future workplace environments.")
    else:
        print("Please enter a valid response (Yes/No).")
        
# Testing the bot
chatbot_ai_careers()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 9:  </b> AI Confidence Level Bot</p></div> 

- This chatbot gauges how confident the user feels about AI tools in education and learning environments (Q9.x).

In [ ]:
def chatbot_confidence_level():
    confidence = input("How confident are you in using AI tools in learning environments? (Very Confident, Confident, Neutral, Unconfident): ").strip().lower()

    if confidence == "very confident":
        print("You seem to be highly confident! Continue exploring new AI tools to stay ahead.")
    elif confidence == "confident":
        print("You have a good level of confidence. Keep using AI tools to enhance your learning.")
    elif confidence == "neutral":
        print("If you're unsure, try starting with simpler AI tools like Grammarly or ChatGPT.")
    elif confidence == "unconfident":
        print("It’s okay to feel unconfident at first. AI tools can be learned gradually.")
    else:
        print("Please enter a valid response (Very Confident, Confident, Neutral, Unconfident).")
        
# Testing the bot
chatbot_confidence_level()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 10:  </b> AI Learning Resources Bot</p></div> 
    
- This bot suggests resources for learning AI based on the user's prior exposure to AI tools (from Q4 and Q5 responses).

In [ ]:
def chatbot_ai_learning_resources():
    exposure = input("How would you describe your exposure to AI tools? (None, Minimal, Moderate, Extensive): ").strip().lower()

    if exposure == "none":
        print("Start with free AI resources like Coursera's 'AI for Everyone' or Google's AI Crash Course.")
    elif exposure == "minimal":
        print("You might want to try interactive AI platforms like Teachable Machine or explore simple AI projects on GitHub.")
    elif exposure == "moderate":
        print("With moderate exposure, you could enhance your knowledge through platforms like FastAI or experimenting with TensorFlow/Keras.")
    elif exposure == "extensive":
        print("With extensive experience, dive into advanced areas like deep learning, reinforcement learning, or AI research papers.")
    else:
        print("Please enter a valid level of exposure (None, Minimal, Moderate, Extensive).")
        
# Testing the bot
chatbot_ai_learning_resources()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 11:  </b> AI Adoption Challenges Bot</p></div> 

- This bot identifies common challenges users face in adopting AI tools based on `Q5.x` responses and offers advice.

In [ ]:
def chatbot_ai_adoption_challenges():
    challenge = input("What is the biggest challenge you face in adopting AI tools? (Lack of Knowledge, Lack of Time, Complexity, Cost): ").strip().lower()

    if challenge == "lack of knowledge":
        print("Consider joining AI communities like Kaggle or AI forums to expand your knowledge through collaboration.")
    elif challenge == "lack of time":
        print("AI learning can be done in small steps. Try spending 30 minutes a week exploring a specific AI topic.")
    elif challenge == "complexity":
        print("Start with simple tools and move step by step. AI doesn't have to be complicated at the beginning.")
    elif challenge == "cost":
        print("Many AI tools are free or open-source. Explore platforms like Google Colab, which offers free cloud computing for AI projects.")
    else:
        print("Please enter a valid challenge (Lack of Knowledge, Lack of Time, Complexity, Cost).")
        
# Testing the bot
chatbot_ai_adoption_challenges()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 12:  </b> AI Tool Recommendation Bot</p></div> 

- This bot recommends AI tools based on specific user needs(from answers in `Q4` and `Q5.x` columns).

In [ ]:
def chatbot_ai_tool_recommendation():
    need = input("What do you need AI tools for? (Text Analysis, Image Processing, Data Analysis, Research): ").strip().lower()

    if need == "text analysis":
        print("You can explore tools like GPT-3 for generating text or analyzing large amounts of written content.")
    elif need == "image processing":
        print("For image processing, try OpenCV or deep learning libraries like TensorFlow and Keras.")
    elif need == "data analysis":
        print("Pandas and Scikit-learn are great for handling data analysis tasks. You might also want to look into AI-powered dashboards like Power BI.")
    elif need == "research":
        print("AI tools like Google Scholar’s AI-enhanced search or semantic search engines can streamline your research process.")
    else:
        print("Please provide a valid need (Text Analysis, Image Processing, Data Analysis, Research).")
        
# Testing the bot
chatbot_ai_tool_recommendation()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 13:  </b> AI Ethics Discussion Bot</p></div> 

- This chatbot explores ethical concerns related to AI based on how the user feels about ethical issues for the `Q9.x` questions.

In [ ]:
def chatbot_ai_ethics_discussion():
    concern = input("What concerns you most about AI ethics? (Bias, Privacy, Accountability, Transparency): ").strip().lower()

    if concern == "bias":
        print("AI bias is a critical issue. It’s important to ensure datasets are representative to avoid unfair outcomes.")
    elif concern == "privacy":
        print("AI privacy concerns are valid. Make sure the AI tools you use comply with privacy laws like GDPR.")
    elif concern == "accountability":
        print("AI accountability is still a growing field. It's vital to clearly define who is responsible for AI decisions.")
    elif concern == "transparency":
        print("Transparency in AI models (such as explainability in decision-making) is essential to build trust in AI technologies.")
    else:
        print("Please enter a valid concern (Bias, Privacy, Accountability, Transparency).")
        
# Testing the bot
chatbot_ai_ethics_discussion()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 14:  </b> Learning AI Difficulty Bot</p></div> 

- This bot gauges the user's difficulty in learning AI tools and provides encouragement or resources to help overvome hurdles(`Q5.x`).

In [ ]:
def chatbot_ai_learning_difficulty():
    difficulty = input("How difficult do you find learning AI tools? (Very Difficult, Difficult, Moderate, Easy): ").strip().lower()

    if difficulty == "very difficult":
        print("AI can be challenging, but start with basic tutorials on YouTube or AI courses on platforms like Coursera to ease the process.")
    elif difficulty == "difficult":
        print("AI requires practice. Try solving small projects or joining communities like Kaggle to learn through experience.")
    elif difficulty == "moderate":
        print("You’re progressing well! Keep learning and experimenting with new tools. Practice makes perfect.")
    elif difficulty == "easy":
        print("Great! Since you find AI learning easy, consider exploring more complex projects or even participating in AI challenges.")
    else:
        print("Please provide a valid difficulty level (Very Difficult, Difficult, Moderate, Easy).")
        
# Testing the bot
chatbot_ai_learning_difficulty()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 15:  </b> Future AI Trends Bot</p></div> 

This bot engages users in a conversation about future AI trends and how they could impact their field of study (Q2 and Q9.x).

In [ ]:
def chatbot_ai_future_trends():
    trend = input("What future AI trend are you most excited about? (AI in Education, AI in Healthcare, AI in Finance, AI in Art): ").strip().lower()

    if trend == "ai in education":
        print("AI in education is transforming how students learn, making personalized learning more accessible. It’s an exciting field!")
    elif trend == "ai in healthcare":
        print("AI in healthcare is making breakthroughs in diagnosis, personalized medicine, and patient care.")
    elif trend == "ai in finance":
        print("AI in finance is reshaping trading strategies, fraud detection, and financial advising. A trend to watch closely!")
    elif trend == "ai in art":
        print("AI is now creating art, music, and designs. The intersection of creativity and AI is a fascinating space!")
    else:
        print("Please enter a valid AI trend (AI in Education, AI in Healthcare, AI in Finance, AI in Art).")
        
# Testing the bot
chatbot_ai_future_trends()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 16:  </b> AI Personalization Bot</p></div> 

- This chatbot explores how the user feels about personalized AI learnng or work environments based on their responses in `Q5.x` and `Q8.x`.

In [ ]:
def chatbot_ai_personalization():
    opinion = input("Do you think AI should be used to personalize learning or work environments? (Yes, No, Neutral): ").strip().lower()

    if opinion == "yes":
        print("Personalization can help create tailored learning experiences and enhance productivity at work.")
    elif opinion == "no":
        print("You might feel that personalization through AI could be intrusive. Privacy and transparency are key in AI applications.")
    elif opinion == "neutral":
        print("It's understandable to be undecided. The impact of personalization in AI can vary depending on its implementation.")
    else:
        print("Please provide a valid response (Yes, No, Neutral).")
        
# Testing the bot
chatbot_ai_personalization()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 17:  </b> AI Career Guidance Bot</p></div> 

- This bot gives career guidance based on the user's degree (`Q2`), field of study(`Q3`), and their level of agreement with statements in `Q5.x`.

In [ ]:
def chatbot_ai_career_guidance():
    degree = input("What is your degree? (Bachelor, Master, PhD): ").strip().lower()
    field_of_study = input("What is your field of study? (e.g., International Economic Relations, Computer Science): ").strip().lower()

    if degree == "bachelor":
        if field_of_study == "international economic relations":
            print("For Bachelor's in International Economic Relations, you could consider careers in economic analysis, international trade consulting, or global business strategy. AI can assist with data-driven decisions.")
        else:
            print(f"For a Bachelor's in {field_of_study}, AI skills in data analysis, automation, and predictive modeling can open many doors.")
    elif degree == "master":
        if field_of_study == "international economic relations":
            print("With a Master's in International Economic Relations, you could explore roles like international economic policy advisor or AI applications in international market forecasting.")
        else:
            print(f"A Master's in {field_of_study} combined with AI expertise can lead to advanced roles like AI research or AI-powered innovations in your domain.")
    elif degree == "phd":
        print("With a PhD, you can delve into AI research, developing new AI models, or leading AI innovation in your field.")
    else:
        print("Please enter a valid degree (Bachelor, Master, PhD).")
        
# Testing the bot
chatbot_ai_career_guidance()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 18:  </b> AI Time Management Bot</p></div> 

- This chatbot helps users manage their time effectively for learning AI, especially if they struggle with balancing work and learning (`Q5.x` responses).

In [ ]:
def chatbot_ai_time_management():
    time_difficulty = input("How often do you find it difficult to find time for learning AI? (Often, Sometimes, Rarely): ").strip().lower()

    if time_difficulty == "often":
        print("It’s great that you want to learn AI, but finding time can be tricky. Try scheduling small, focused sessions (15-30 minutes) in your weekly calendar to slowly build your AI skills.")
    elif time_difficulty == "sometimes":
        print("AI learning can be demanding, but finding some time each week can help you progress. Consider learning on weekends or during free time with bite-sized content like YouTube videos.")
    elif time_difficulty == "rarely":
        print("You seem to be managing your time well. Keep exploring new topics and experimenting with AI projects whenever you get a chance!")
    else:
        print("Please enter a valid response (Often, Sometimes, Rarely).")
        
# Testing the bot
chatbot_ai_time_management()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 19:  </b> AI Research Bot</p></div> 

- This chatbot helps users identify potential AI research topics based on their field of study (`Q3`) and responses to ethical issues(`Q9.x`).

In [ ]:
def chatbot_ai_research_suggestions():
    field_of_study = input("What is your field of study? (e.g., International Economic Relations, Computer Science): ").strip().lower()
    ethical_concern = input("What ethical concern in AI are you interested in exploring? (Bias, Privacy, Accountability, Transparency): ").strip().lower()

    if ethical_concern == "bias":
        print(f"In {field_of_study}, research how bias in AI algorithms can affect decision-making processes in international trade, global policies, or economic forecasts.")
    elif ethical_concern == "privacy":
        print(f"Privacy concerns in AI are crucial in {field_of_study}. You can explore how AI technologies impact privacy in global business transactions or international data handling.")
    elif ethical_concern == "accountability":
        print(f"Explore how AI accountability is managed in {field_of_study}, such as responsibility for AI-driven decisions in economic markets or international relations.")
    elif ethical_concern == "transparency":
        print(f"Transparency is key to AI. In {field_of_study}, you can explore how transparent AI models can improve international business transparency or economic predictions.")
    else:
        print("Please enter a valid ethical concern (Bias, Privacy, Accountability, Transparency).")
        
# Testing the bot
chatbot_ai_research_suggestions()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 20:  </b> AI Learning Style Bot</p></div> 

- This chatbot determines the user's AI learning style based o how they respond to structured learning or self-paced learning (`Q5.x`)

In [ ]:
def chatbot_ai_learning_style():
    structured_learning = input("Do you prefer structured learning (courses, curriculums) or self-paced exploration? (Structured, Self-paced): ").strip().lower()

    if structured_learning == "structured":
        print("If you prefer structured learning, try enrolling in comprehensive AI courses on platforms like Coursera or edX, where you can follow a set curriculum with guided learning.")
    elif structured_learning == "self-paced":
        print("For self-paced learning, explore free resources like GitHub repositories, AI blogs, or YouTube tutorials to learn AI at your own speed.")
    else:
        print("Please enter a valid learning preference (Structured, Self-paced).")
        
# Testing the bot
chatbot_ai_learning_style()


# <div style="color:white;display:inline-block;border-radius:5px;background-color:#007BA7;font-family:cursive;overflow:hidden"><p style="padding:8px;color:white;overflow:hidden;font-size:90%;letter-spacing:0.5px;margin:0"><b>Chatbot 21:  </b> AI Skill Building Bot</p></div> 

- This chatbot suggests specific skill users should build to improve their AI proficiency based on their current experience level(`Q4` and `Q5.x`).

In [ ]:
def chatbot_ai_skill_building():
    experience_level = input("What is your current experience level with AI? (Beginner, Intermediate, Advanced): ").strip().lower()

    if experience_level == "beginner":
        print("Start with basic skills like Python programming, data handling (Pandas), and learning simple AI concepts like classification and regression.")
    elif experience_level == "intermediate":
        print("Enhance your skills by diving into neural networks, deep learning, and working with popular AI libraries like TensorFlow and PyTorch.")
    elif experience_level == "advanced":
        print("Focus on mastering AI frameworks, optimizing deep learning models, or exploring research-level AI topics such as reinforcement learning or generative models.")
    else:
        print("Please enter a valid experience level (Beginner, Intermediate, Advanced).")
        
# Testing the bot
chatbot_ai_skill_building()
